In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## preprocessing and data cleaning

In [17]:
nasal = pd.read_csv("Flow - 30-05-2024.txt", delimiter='\t')
thor = pd.read_csv("Thorac - 30-05-2024.txt", delimiter='\t')
spo = pd.read_csv("SPO2 - 30-05-2024.txt", delimiter='\t')

In [18]:
nasal = nasal.iloc[5: , :]
thor = thor.iloc[5:, :]
spo = spo.iloc[5:, :]

In [19]:
nasal['Time'] = nasal['Signal Type: Flow_TH_Type'].str.split(';').str[0]
nasal['Value'] = nasal['Signal Type: Flow_TH_Type'].str.split(';').str[1]

In [20]:
thor['Time'] = thor['Signal Type: Sum RIPs-Reference'].str.split(';').str[0]
thor['Value'] = thor['Signal Type: Sum RIPs-Reference'].str.split(';').str[1]

In [21]:
spo['Time'] = spo['Signal Type: SPO2_Type'].str.split(';').str[0]
spo['Value'] = spo['Signal Type: SPO2_Type'].str.split(';').str[1]

In [22]:
nasal_origin = nasal.copy()
thor_origin = thor.copy()
spo_origin = spo.copy()

In [23]:
nasal.drop(['Signal Type: Flow_TH_Type'], axis=1, inplace=True)
thor.drop(['Signal Type: Sum RIPs-Reference'], axis=1, inplace=True)
spo.drop(['Signal Type: SPO2_Type'], axis=1, inplace=True)

In [24]:
nasal = nasal.set_index('Time')
thor = thor.set_index('Time')
spo = spo.set_index('Time')

In [25]:
nasal.index = pd.to_datetime(nasal.index, format="%d.%m.%Y %H:%M:%S,%f")
nasal['Value'] = pd.to_numeric(nasal['Value'].astype(str).str.strip(), errors='coerce')

In [26]:
thor.index = pd.to_datetime(thor.index, format="%d.%m.%Y %H:%M:%S,%f")
thor['Value'] = pd.to_numeric(thor['Value'].astype(str).str.strip(), errors='coerce')

In [27]:
spo.index = pd.to_datetime(spo.index, format="%d.%m.%Y %H:%M:%S,%f")
spo['Value'] = pd.to_numeric(spo['Value'].astype(str).str.strip(), errors='coerce')

In [28]:
events = pd.read_csv("Flow Events - 30-05-2024.txt", delimiter='\t')
events = events.iloc[3: , :]
events['Time'] = events[r'Signal ID: FlowD\flow'].str.split(';').str[0]
events['Value'] = events[r'Signal ID: FlowD\flow'].str.split(';').str[1]
events['Disorder'] = events[r'Signal ID: FlowD\flow'].str.split(';').str[2]
events['Stage'] = events[r'Signal ID: FlowD\flow'].str.split(';').str[3]
events_origin = events


events[['start_str', 'end_str']] = events['Time'].str.split('-', expand=True)
events['date_part'] = events['start_str'].str.split(' ').str[0]
events['end_str'] = events['date_part'] + ' ' + events['end_str']
events['start_time'] = pd.to_datetime(events['start_str'], format="%d.%m.%Y %H:%M:%S,%f")
events['end_time'] = pd.to_datetime(events['end_str'], format="%d.%m.%Y %H:%M:%S,%f")
events.drop([r'Signal ID: FlowD\flow', 'Time', 'start_str', 'end_str', 'date_part'], axis=1, inplace=True)


## data window creation and actual viz

In [32]:
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

window = pd.Timedelta(minutes=5)

start_time = max(nasal.index[0], thor.index[0], spo.index[0])
end_time = min(nasal.index[-1], thor.index[-1], spo.index[-1])

current_time = start_time

with PdfPages("AP01_visualization.pdf") as pdf:

    while current_time < end_time:

        window_end = current_time + window

        seg_flow = nasal.loc[current_time:window_end]
        seg_thorac = thor.loc[current_time:window_end]
        seg_spo2 = spo.loc[current_time:window_end]

        fig, axs = plt.subplots(3, 1, figsize=(18, 9), sharex=True)

        # --- Plot signals ---
        axs[0].plot(seg_flow.index, seg_flow['Value'], color='tab:blue', linewidth=0.8)
        axs[0].set_ylabel("Flow")

        axs[1].plot(seg_thorac.index, seg_thorac['Value'], color='tab:orange', linewidth=0.8)
        axs[1].set_ylabel("Thorac")

        axs[2].plot(seg_spo2.index, seg_spo2['Value'], color='tab:green', linewidth=1.0)
        axs[2].set_ylabel("SpO₂")
        axs[2].set_xlabel("Time")

        # --- Add grid ---
        for ax in axs:
            ax.grid(True, alpha=0.3)
            ax.set_xlim(current_time, window_end)

        # --- Highlight Events ---
        events_in_window = events[
            (events['start_time'] <= window_end) &
            (events['end_time'] >= current_time)
        ]

        for _, row in events_in_window.iterrows():

            event_start = max(row['start_time'], current_time)
            event_end = min(row['end_time'], window_end)

            if row['Disorder'] == "Obstructive Apnea":
                color = 'red'
            elif row['Disorder'] == "Hypopnea":
                color = 'yellow'
            else:
                color = 'purple'

            for ax in axs:
                ax.axvspan(event_start, event_end, color=color, alpha=0.3)

        # --- Improve time ticks ---
        locator = mdates.MinuteLocator(interval=1)   # tick every 1 minute
        formatter = mdates.DateFormatter('%H:%M:%S')

        axs[-1].xaxis.set_major_locator(locator)
        axs[-1].xaxis.set_major_formatter(formatter)

        # --- Title ---
        fig.suptitle(
            f"AP01 | {current_time.strftime('%H:%M')} - {window_end.strftime('%H:%M')}",
            fontsize=14,
            fontweight='bold'
        )

        plt.tight_layout(rect=[0, 0, 1, 0.96])

        pdf.savefig(fig)
        plt.close(fig)

        current_time = window_end